In [1]:

# !pip install pandas
# !pip install pinecone
# !pip unstall openai
# !pip install python-dotenv
# !pip install scikit-learn

In [2]:
from openai import OpenAI
import os
import json
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
!pip install sklearn

  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'error'


  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
      
      More information is available at
      https://github.com/scikit-learn/sklearn-pypi-packag

In [ ]:
from openai import OpenAI
import os

MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"

client = OpenAI(
    api_key=os.getenv("RUNPOD_API_KEY"),
    base_url=os.getenv("RUNPOD_CHATBOT_URI"),
)

In [5]:
def get_chatbot_response(client,model_name,messages,temperature=0):
    input_messages = []
    for message in messages:
        input_messages.append({"role": message["role"], "content": message["content"]})

    response = client.chat.completions.create(
        model=model_name,
        messages=input_messages,
        temperature=temperature,
        top_p=0.8,
        max_tokens=2000,
    ).choices[0].message.content
    
    return response

In [6]:
!pip show openai


Name: openai
Version: 1.101.0
Summary: The official Python library for the openai API
Home-page: https://github.com/openai/openai-python
Author: 
Author-email: OpenAI <support@openai.com>
License: Apache-2.0
Location: C:\Users\lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages
Requires: anyio, distro, httpx, jiter, pydantic, sniffio, tqdm, typing-extensions
Required-by: litellm


GET LLM RESPONSE

In [8]:
messages = [{'role':'user','content':"What's the capital of Italy?"}]
response = get_chatbot_response(client,MODEL_NAME,messages)
print(response)

KeyboardInterrupt: 

# Prompt engineering

## Structured output

In [ ]:
system_prompt = """
You are a helpful assistant that answers questions about capitals of countries.

Your output MUST be valid JSON. 
Always use double quotes for property names and string values. 
Do not include anything other than the JSON array.

Format:
[
  {
    "country": "the country name",
    "capital": "the capital city"
  }
]
"""

messages = [{'role':'system','content':system_prompt}]
messages.append({'role':'user','content':"What's the capital of Italy?"})
response = get_chatbot_response(client,MODEL_NAME,messages)
print(response)

[
  {
    "country": "Italy",
    "capital": "Rome"
  }
]


In [9]:
json_response = json.loads(response)
json_response

[{'country': 'Italy', 'capital': 'Rome'}]

In [ ]:
user_prompt="""
Give me the capital of countries:
```
1.Germany
2.Australia
3.America
```
"""

messages=[{"role":"system", "content":system_prompt}]
messages.append({"role":"user", "content":user_prompt})
response=get_chatbot_response(client, MODEL_NAME,messages)
print(response)



# Chain Of Thought

In [11]:
user_prompt = """
Calculate the result of this equation: 1+3

Your output should be in a structured json format exactly like the one below. You are not allowed to write anything other than the json object:
{
    result: The final number resulted from calculating the equation above
}
"""

messages = [{'role':'user','content':user_prompt}]
response = get_chatbot_response(client,MODEL_NAME,messages)
print(response)

{
    "result": 4
}


In [12]:
user_prompt = """
Calculate the result of this equation: 259/2*8654+91072*33-12971

Your output should be in a structured json format exactly like the one bellow. You are not allowed to write anything other than the json object:
{
    steps: This is where you solve the equation bit by bit following the BEDMAS order of operations. You need to show your work and calculate each step leading to final result. Feel free to write here in free text. 
    result: The final number resulted from calculating the equation above
}
"""

messages = [{'role':'user','content':user_prompt}]
response = get_chatbot_response(client,MODEL_NAME,messages)

# RAG - Retrieval Augmented Generation

In [ ]:
user_prompt = """
What's new in iphone 16?
"""

messages = [{'role':'user','content':user_prompt}]
response = get_chatbot_response(client,MODEL_NAME,messages)
print(response)

In [ ]:
iphone_16 = """
The iPhone 16 introduces several exciting updates, making it one of Apple's most advanced smartphones to date. It features a larger 6.1-inch display for the base model and a 6.7-inch screen for the iPhone 16 Plus, with thinner bezels and a more durable Ceramic Shield. The iPhone 16 Pro and Pro Max boast even larger displays, measuring 6.3 and 6.9 inches respectively, offering the thinnest bezels seen on any Apple product so far.

Powered by the new A18 chip (A18 Pro for the Pro models), these phones deliver significant performance improvements, with enhanced neural engine capabilities, faster GPU for gaming, and machine learning tasks. The camera systems are also upgraded, with the base iPhone 16 sporting a dual-camera setup with a 48MP main sensor. The Pro models offer a 48MP Ultra Wide and 5x telephoto camera, enhanced by Apple’s "Camera Control" button for more flexible photography options.

Apple also introduced advanced audio features like "Audio Mix," which uses machine learning to separate background sounds from speech, allowing for more refined audio capture during video recording. Battery life has been extended, especially in the iPhone 16 Pro Max, which is claimed to have the longest-lasting battery of any iPhone 
9TO5MAC

APPLEMAGAZINE
.

Additionally, Apple has switched to USB-C for faster charging and data transfer, and the Pro models now support up to 2x faster video encoding. The starting prices remain consistent with previous generations, with the iPhone 16 starting at $799, while the Pro models start at $999
"""

In [ ]:
user_prompt = f"""
{iphone_16}

What's new in iphone 16?
"""

messages = [{'role':'user','content':user_prompt}]
response = get_chatbot_response(client,model_name,messages)
print(response)

In [ ]:
samsung_s23 = """
The Samsung Galaxy S23 brings some incremental but notable upgrades to its predecessor, the Galaxy S22. It features the Snapdragon 8 Gen 2 processor, a powerful chip optimized for the S23 series, delivering enhanced performance, especially for gaming and multitasking. This chip ensures top-tier speed and efficiency across all models, from the base S23 to the larger S23+ and S23 Ultra​
STUFF

TECHRADAR
.

In terms of design, the S23's camera module has been streamlined by removing the raised metal contour around the cameras, creating a cleaner, sleeker look. It also sports the same 6.1-inch 120Hz AMOLED display, protected by tougher Gorilla Glass Victus 2, making it more resistant to scratches and drops​
TECHRADAR
.

The S23 Ultra stands out with its 200MP main camera, offering impressive photo clarity, especially in low-light conditions. The selfie camera across the series has been updated to a 12MP sensor, resulting in sharper selfies. The Ultra model also includes productivity tools such as the S-Pen, which remains an essential feature for note-taking and creative tasks​
STUFF

TECHRADAR
.

Battery life is solid, with the S23 Ultra featuring a 5000mAh battery that lasts comfortably through a day of heavy use. However, charging speeds still lag behind some competitors, with 45W wired charging, which is slower than other brands offering up to 125W charging​
STUFF
.

Overall, the Galaxy S23 series enhances performance, durability, and camera quality, making it a strong contender for users seeking a high-performance flagship.
"""

In [ ]:
data = [iphone_16,samsung_s23]

In [24]:
user_prompt = """What's new in iphone 16?"""

In [ ]:
embedding_client = OpenAI(
        api_key=os.getenv("RUNPOD_API_KEY"),
        base_url=os.getenv("RUNPOD_EMBEDDING_URL")
    )

In [40]:
def get_embedding(embedding_client,model_name,text_input):
    output = embedding_client.embeddings.create(input = text_input,model=model_name)
    
    embedings = []
    for embedding_object in output.data:
        embedings.append(embedding_object.embedding)

    return embedings


In [ ]:
user_prompt_embeddings = get_embedding(embedding_client,MODEL_NAME,user_prompt)

In [43]:
print(user_prompt_embeddings)

[[-0.05107622966170311, -0.03489547595381737, 0.06362394988536835, -0.008907047100365162, -0.03724626451730728, -0.0420089028775692, 0.015967046841979027, 0.031155584380030632, -0.012799587100744247, 0.04243631660938263, 0.05556409806013107, 0.013288062997162342, -0.07217226922512054, 0.00139673484954983, 0.09635180979967117, 0.028697941452264786, 0.11521918326616287, -0.14751963317394257, -0.07907198369503021, 0.016165489330887794, -0.03666619956493378, -0.014616105705499649, -0.039841290563344955, -0.030025985091924667, 0.030025985091924667, 0.061517395079135895, 0.0010771268280223012, -0.012753792107105255, -0.019157402217388153, -0.13848282396793365, -0.000405234401114285, -0.006754701491445303, 0.01666923053562641, 0.026835627853870392, 0.00040475736022926867, -0.038497984409332275, -0.0031502859201282263, 0.03492600470781326, -0.009632128290832043, 0.06380712240934372, 0.029629098251461983, 0.09103964269161224, -0.10141974687576294, -0.024194806814193726, 0.046832598745822906, 0.

In [ ]:
user_prompt_embeddings = user_prompt_embeddings[0]

In [ ]:
data_embeddings = [get_embedding(embedding_client,MODEL_NAME,x)[0] for x in data]

In [ ]:
data_embeddings = [get_embedding(embedding_client,MODEL_NAME,x)[0] for x in data]

In [ ]:
data_similaraty_scores = cosine_similarity([user_prompt_embeddings], data_embeddings)

In [ ]:
data_similaraty_scores

In [ ]:
closest_entry_index=data_similaraty_scores.argmax()
closest_entry_index

In [ ]:
data[closest_entry_index]

In [ ]:
user_prompt_with_data = f"""
{data[closest_entry_index]}

{user_prompt}
""" 

In [ ]:
messages = [{'role':'user','content':user_prompt_with_data}]
response = get_chatbot_response(client,model_name,messages)
print(response)